# Cosmic Sort 2026: STAR / GALAXY / QSO

Reproduces our submission from the competition data only.

* **Data**: only the provided `train.csv` / `test.csv`. No external data, no test lookups.
* **Redshift**: never a model input. It is used only as a *training target* (a photo-z model and an
  auxiliary neural-net head), which the coordinator confirmed is allowed.
* **Validation**: one fixed 5-fold stratified split (seed 42) for every model; the blend is scored with nested CV.
* **Pipeline**: label-free features (incl. sky-neighbour, colour-density and noise-aware colour-mixture
  groups) -> clean per-fold photo-z features (built without the validation fold's labels) ->
  LightGBM / XGBoost (10-fold) and three neural nets (10-fold) -> calibrated blend -> `submission.csv`.
* Feature set `raw,color,flux,morph,errres,sky,local,dens,nmix,spec`; tuned parameters: lgb, xgb.

**Progress**: every step prints `STEP k/N`, per-fold lines with time left, and a progress table.
The same table is saved as `progress.md` in the output folder.

In [ ]:
# 'full' = final submission (~2h on GPU T4 x2) | 'fast' = quick check, 1 seed each (~20 min)
MODE = 'full'

In [ ]:
RECIPES = {
 "full": [
  [
   "Clean photo-z features (nested, per fold)",
   "src/nested.py",
   6
  ],
  [
   "LightGBM + clean photo-z, 20-fold, 1 seed",
   "src/train_gbdt.py --model lgb --features best --nested --seeds 0 --lr 0.03 --nfolds 20 --name lgb_best",
   20
  ],
  [
   "XGBoost (GPU) + clean photo-z, 10-fold, 1 seed",
   "src/train_gbdt.py --model xgb --features best --nested --seeds 0 --lr 0.03 --nfolds 10 --name xgb_best --device gpu",
   5
  ],
  [
   "CatBoost (GPU) + clean photo-z, 10-fold, 1 seed",
   "src/train_gbdt.py --model cat --features best --nested --seeds 0 --lr 0.06 --nfolds 10 --name cat_best --device gpu",
   6
  ],
  [
   "Neural net 256x4, 20-fold, 2 seeds",
   "src/train_mlp.py --features best,-dens,-nmix,-spec --seeds 0,1 --hidden 256 --layers 4 --aux 0 --nfolds 20 --name mlp_base",
   26
  ],
  [
   "Neural net 512x3 + redshift head, 10-fold, 1 seed",
   "src/train_mlp.py --features best,-dens,-nmix,-spec --seeds 0 --aux 0.2 --nfolds 10 --name mlp_aux",
   5
  ]
 ],
 "fast": [
  [
   "Clean photo-z features (nested, per fold)",
   "src/nested.py",
   8
  ],
  [
   "LightGBM + clean photo-z, 1 seed",
   "src/train_gbdt.py --model lgb --features best --nested --seeds 0 --lr 0.05 --name f_lgb_best",
   3
  ],
  [
   "XGBoost (GPU) + clean photo-z, 1 seed",
   "src/train_gbdt.py --model xgb --features best --nested --seeds 0 --lr 0.05 --name f_xgb_best --device gpu",
   2
  ],
  [
   "Neural net 256x4, 1 seed",
   "src/train_mlp.py --features best,-dens,-nmix --seeds 0 --hidden 256 --layers 4 --aux 0 --name f_mlp_base",
   3
  ]
 ]
}
PREP_MIN, ENS_MIN = 5, 5

import glob, json, os, re, shlex, subprocess, sys, time
T0 = time.time()
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    cands = [p for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'test.csv'))]
    assert cands, 'Competition data not found: right panel -> + Add Input -> Competitions -> Cosmic Sort 2026'
    DATA, WD = os.path.dirname(cands[0]), '/kaggle/working'
else:                                    # local test of this notebook
    DATA, WD = os.environ['CS_NB_DATA'], os.environ['CS_NB_WORKDIR']
os.makedirs(os.path.join(WD, 'src'), exist_ok=True)
os.chdir(WD)
os.environ.update(CS_ROOT=WD, CS_DATA=DATA, CS_WORK=os.path.join(WD, 'work'), CS_SUBS=os.path.join(WD, 'subs'),
                  PYTHONWARNINGS='ignore', PYTHONUNBUFFERED='1')

import torch, lightgbm, xgboost, catboost, sklearn
GPU = torch.cuda.is_available()
print('data folder :', DATA, sorted(os.listdir(DATA)))
print('CPU cores   :', os.cpu_count(), '| GPU:', torch.cuda.get_device_name(0) if GPU else 'NONE')
print('versions    : lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| catboost',
      catboost.__version__, '| torch', torch.__version__, '| sklearn', sklearn.__version__)
if not GPU:
    print('\n*** WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2. On CPU only this takes much longer. ***\n')

PLAN = [('Features, photo-z, neighbour features', 'src/prep.py', PREP_MIN)] + \
       [(l, c if GPU else c.replace('--device gpu', '--device cpu'), m) for l, c, m in RECIPES[MODE]] + \
       [('Blend + calibrate + write submission.csv', 'src/ensemble.py --tag kaggle_' + MODE + ' --out ' +
         WD.replace(os.sep, '/') + '/submission.csv', ENS_MIN)]
STATUS = {i: dict(label=l, est=m, state='pending', minutes=None, cv=None) for i, (l, c, m) in enumerate(PLAN)}
NOISE = re.compile(r'Warning|warnings\.warn|^\s*$')


def mm(s):
    return f'{int(s // 3600)}h{int(s % 3600 // 60):02d}m' if s >= 3600 else f'{int(s // 60)}m{int(s % 60):02d}s'


def write_progress(final=False):
    el = time.time() - T0
    left = sum(s['est'] for s in STATUS.values() if s['state'] in ('pending', 'running')) * 60
    icon = dict(pending='waiting', running='RUNNING', done='done', failed='FAILED')
    rows = [f"| {i + 1} | {s['label']} | {icon[s['state']]} | "
            f"{(str(round(s['minutes'], 1)) + ' min') if s['minutes'] is not None else '~' + str(s['est']) + ' min (est.)'} | "
            f"{s['cv'] if s['cv'] else ''} |" for i, s in STATUS.items()]
    txt = (f"# Cosmic Sort run ({MODE}) - {'FINISHED' if final else 'in progress'}\n\n"
           f"Elapsed **{mm(el)}** | estimated left **{mm(left)}** | Kaggle limit 3h00m\n\n"
           "| # | Step | State | Time | CV log loss |\n|---|---|---|---|---|\n" + '\n'.join(rows) + '\n')
    open(os.path.join(WD, 'progress.md'), 'w').write(txt)
    return txt


def last_cv(name):
    f = os.path.join(WD, 'work', 'results.jsonl')
    if not os.path.exists(f):
        return None
    for line in reversed(open(f).read().splitlines()):
        r = json.loads(line)
        if r['name'] == name:
            return f"{r['cv']:.5f}"


def stream(cmd):
    p = subprocess.Popen([sys.executable, '-u'] + shlex.split(cmd), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if not NOISE.search(line):
            print(line, end='', flush=True)
    return p.wait()


def run_step(i):
    label, cmd, est = PLAN[i]
    s = STATUS[i]
    s['state'] = 'running'
    write_progress()
    print('\n' + '=' * 100 + f'\nSTEP {i + 1}/{len(PLAN)}: {label}   (est. ~{est} min, total elapsed {mm(time.time() - T0)})'
          + f'\n  $ python {cmd}\n' + '=' * 100, flush=True)
    t = time.time()
    rc = stream(cmd)
    if rc != 0 and '--device gpu' in cmd:
        print('\n*** GPU run failed, retrying this step on CPU ***\n', flush=True)
        cmd = cmd.replace('--device gpu', '--device cpu')
        rc = stream(cmd)
    if rc != 0 and '--nested' in cmd:
        print('\n*** nested photo-z features missing, retrying this step without them ***\n', flush=True)
        rc = stream(cmd.replace(' --nested', ''))
    s['minutes'] = (time.time() - t) / 60
    s['state'] = 'done' if rc == 0 else 'failed'
    m = re.search(r'--name (\S+)', cmd)
    s['cv'] = last_cv(m.group(1)) if (m and rc == 0) else None
    if 'ensemble.py' in cmd and rc == 0:
        e = open(os.path.join(WD, 'work', 'ensembles.jsonl')).read().splitlines()[-1]
        s['cv'] = f"{json.loads(e)['nested_cv']:.5f} (blend)"
    print(f"\n>>> STEP {i + 1} {s['state'].upper()} in {s['minutes']:.1f} min" + (f" | CV {s['cv']}" if s['cv'] else ''))
    print(write_progress(), flush=True)
    return rc

os.environ['CS_PREP_LEARNED'] = '0'   # V8: skip the unused old learned-feature step

In [ ]:
%%writefile src/common.py
"""Shared paths, constants and helpers for the Cosmic Sort pipeline."""
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(os.environ.get('CS_ROOT', Path(__file__).resolve().parents[1]))
DATA = Path(os.environ.get('CS_DATA', ROOT / 'data'))
WORK = Path(os.environ.get('CS_WORK', ROOT / 'work'))
PREDS = Path(os.environ.get('CS_PREDS', WORK / 'preds'))
PARAMS = WORK / 'params'
SUBS = Path(os.environ.get('CS_SUBS', ROOT / 'subs'))
for _d in (WORK, PREDS, PARAMS, SUBS):
    _d.mkdir(parents=True, exist_ok=True)

CLASSES = ['STAR', 'GALAXY', 'QSO']          # column order of every probability array
BANDS = ['u', 'g', 'r', 'i', 'z']
N_FOLDS = 5
FOLD_SEED = 42
N_JOBS = int(os.environ.get('SLURM_CPUS_PER_TASK', os.cpu_count() or 4))
SMOKE = os.environ.get('CS_SMOKE', '0') == '1'   # tiny run to test the whole pipeline end to end
EPS = 1e-15


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


def fmt_secs(s):
    s = int(s)
    return f'{s // 3600}h{s % 3600 // 60:02d}m' if s >= 3600 else f'{s // 60}m{s % 60:02d}s'


class Progress:
    """Prints '[name] 3/15 <msg> | elapsed 2m10s | ~4m20s left' after each unit of work."""

    def __init__(self, name, total):
        self.name, self.total, self.done, self.t0 = name, total, 0, time.time()

    def step(self, msg=''):
        self.done += 1
        el = time.time() - self.t0
        left = el / self.done * (self.total - self.done)
        log(f'[{self.name}] {self.done}/{self.total} {msg} | elapsed {fmt_secs(el)} | ~{fmt_secs(left)} left')


def load_raw():
    tr = pd.read_csv(DATA / 'train.csv')
    te = pd.read_csv(DATA / 'test.csv')
    if SMOKE:
        tr = tr.sample(6000, random_state=0).reset_index(drop=True)
        te = te.sample(2000, random_state=0).reset_index(drop=True)
    return tr, te


def labels(tr):
    return tr['class'].map({c: i for i, c in enumerate(CLASSES)}).values.astype(np.int64)


def mlogloss(y, p):
    p = np.clip(p / p.sum(1, keepdims=True), EPS, 1)
    return float(-np.log(p[np.arange(len(y)), y]).mean())


def load_work():
    """Features built by prep.py: train/test frames, labels, fold ids, feature groups."""
    tr = pd.read_pickle(WORK / 'feat_train.pkl')
    te = pd.read_pickle(WORK / 'feat_test.pkl')
    y = np.load(WORK / 'y.npy')
    fold = np.load(WORK / 'fold.npy')
    groups = json.loads((WORK / 'feature_groups.json').read_text())
    return tr, te, y, fold, groups


def split_folds(fold, y, n):
    """n=5: the fixed folds. n=10 or 20: each fixed fold split into n/5 parts (stratified, seed 42), so
    models train on 90% / 95% of rows while every prediction is still out-of-fold and the 5-fold blend
    still works. (n=10 gives exactly the same split as before.)"""
    if n == N_FOLDS:
        return fold
    k = n // N_FOLDS
    assert n % N_FOLDS == 0 and k >= 2, 'nfolds must be 5, 10, 15, 20, ...'
    from sklearn.model_selection import StratifiedKFold
    out = np.zeros_like(fold)
    for f in range(N_FOLDS):
        idx = np.where(fold == f)[0]
        for j, (_, b) in enumerate(StratifiedKFold(k, shuffle=True, random_state=FOLD_SEED).split(idx, y[idx])):
            out[idx[b]] = k * f + j
    return out


def select(groups, spec):
    """spec: 'all', 'best' (chosen by ablate.py), or a comma list of group names; '-name' removes a
    group (e.g. 'all,-knn_color')."""
    if spec.startswith('best'):
        f = WORK / 'best_features.txt'
        spec = (f.read_text().strip() if f.exists() else 'all') + spec[4:]
    names = list(groups) if spec.startswith('all') else []
    for tok in spec.split(','):
        tok = tok.strip()
        if tok in ('', 'all'):
            continue
        if tok.startswith('-'):
            names.remove(tok[1:])
        else:
            names.append(tok)
    cols = []
    for n in names:
        cols += [c for c in groups[n] if c not in cols]
    return cols


def save_preds(name, oof, test, y, info=None):
    score = mlogloss(y, oof)
    np.savez_compressed(PREDS / f'{name}.npz', oof=oof.astype(np.float32), test=test.astype(np.float32))
    row = dict(name=name, cv=round(score, 6), time=time.strftime('%Y-%m-%d %H:%M'), **(info or {}))
    with open(WORK / 'results.jsonl', 'a') as f:
        f.write(json.dumps(row) + '\n')
    log(f'SAVED {name}: CV logloss {score:.5f}')
    return score

In [ ]:
%%writefile src/features.py
"""Label-free features. Statistics (error-vs-magnitude curves, sky density) use train+test together,
which is allowed: it is competition data and no labels are involved."""
import numpy as np
import pandas as pd
from sklearn.neighbors import KDTree, NearestNeighbors

from common import BANDS, N_JOBS

LOCAL_K = 300

RAW = BANDS + [f'err_{b}' for b in BANDS] + ['size_r', 'gal_l', 'gal_b']
PAIRS = [(a, b) for i, a in enumerate(BANDS) for b in BANDS[i + 1:]]          # all 10 colours
ADJ = list(zip(BANDS, BANDS[1:]))


def _binned_curve(x, v, nbins=60):
    """Median of v as a smooth function of x (quantile bins + linear interpolation)."""
    q = np.unique(np.quantile(x, np.linspace(0, 1, nbins + 1)))
    idx = np.clip(np.searchsorted(q, x, side='right') - 1, 0, len(q) - 2)
    cx = np.array([np.median(x[idx == k]) for k in range(len(q) - 1)])
    cv = np.array([np.median(v[idx == k]) for k in range(len(q) - 1)])
    return np.interp(x, cx, cv)


def _binned_quantile(x, v, qs, nbins=60):
    q = np.unique(np.quantile(x, np.linspace(0, 1, nbins + 1)))
    idx = np.clip(np.searchsorted(q, x, side='right') - 1, 0, len(q) - 2)
    cx = np.array([np.median(x[idx == k]) for k in range(len(q) - 1)])
    out = []
    for qq in qs:
        cv = np.array([np.quantile(v[idx == k], qq) for k in range(len(q) - 1)])
        out.append(np.interp(x, cx, cv))
    return out


def sky_vectors(df):
    l, b = np.radians(df['gal_l'].values), np.radians(df['gal_b'].values)
    return np.c_[np.cos(b) * np.cos(l), np.cos(b) * np.sin(l), np.sin(b)]


def chord(deg):
    return 2 * np.sin(np.radians(deg) / 2)


def build(tr, te):
    """Returns (train_features, test_features, groups) where groups maps group name -> columns."""
    df = pd.concat([tr[RAW], te[RAW]], ignore_index=True)
    n_tr = len(tr)
    F, groups = {}, {}

    def add(group, name, values):
        F[name] = np.asarray(values, dtype=np.float32)
        groups.setdefault(group, []).append(name)

    for c in RAW:
        add('raw', c, df[c])

    # colours, colour errors and significance
    for a, b in PAIRS:
        add('color', f'{a}-{b}', df[a] - df[b])
    for a, b in ADJ:
        e = np.hypot(df[f'err_{a}'], df[f'err_{b}'])
        add('color', f'e_{a}-{b}', e)
        add('color', f'sig_{a}-{b}', (df[a] - df[b]) / e)
    add('color', 'mean_mag', df[BANDS].mean(1))
    add('color', 'ug_x_gr', (df.u - df.g) * (df.g - df.r))
    add('color', 'gr_x_ri', (df.g - df.r) * (df.r - df.i))

    # fluxes: shape of the spectrum independent of brightness, and per-band SNR
    flux = np.stack([10 ** (-0.4 * (df[b].values - 22.5)) for b in BANDS], 1)
    tot = flux.sum(1, keepdims=True)
    for k, b in enumerate(BANDS):
        add('flux', f'sed_{b}', flux[:, k] / tot[:, 0])
        add('flux', f'snr_{b}', 1.0857 / df[f'err_{b}'])
    add('flux', 'log_tot_flux', np.log10(tot[:, 0]))
    add('flux', 'u_drop', (df.u - df.g) - (df.g - df.r))          # high-redshift quasar u dropout

    # morphology: is the object bigger than a point source of the same brightness?
    ls = np.log(df['size_r'].values)
    add('morph', 'log_size', ls)
    add('morph', 'sb_r', df['r'].values + 5 * np.log10(df['size_r'].values))
    p10, p25, p50 = _binned_quantile(df['r'].values, ls, [0.10, 0.25, 0.50])
    add('morph', 'size_res', (ls - p25) / np.maximum(p50 - p10, 1e-3))
    add('morph', 'size_res_raw', ls - p25)

    # error-vs-brightness residuals: extended objects have larger errors at fixed magnitude
    for b in BANDS:
        le = np.log(df[f'err_{b}'].values)
        add('errres', f'eres_{b}', le - _binned_curve(df[b].values, le))
    E = np.stack([F[f'eres_{b}'] for b in BANDS], 1)
    add('errres', 'eres_mean', E.mean(1))
    add('errres', 'eres_gri', E[:, 1:4].mean(1))

    # sky position and local object density (Milky Way plane = many stars)
    V = sky_vectors(df)
    add('sky', 'abs_b', df['gal_b'].abs())
    add('sky', 'sin_b', V[:, 2])
    add('sky', 'vx', V[:, 0])
    add('sky', 'vy', V[:, 1])
    tree = KDTree(V)
    for deg in (0.25, 1.0):
        add('sky', f'dens_{deg}', np.log1p(tree.query_radius(V, chord(deg), count_only=True)))

    # 'local': compare each object with its LOCAL_K nearest objects on the sky (train+test, no labels).
    # Removes field-to-field changes in seeing (sizes), sky depth (errors) and Milky Way dust (colours).
    # Experiment 31 Oct: LightGBM CV 0.15713 -> 0.15599 (K=300).
    _, idx = NearestNeighbors(n_neighbors=LOCAL_K + 1, n_jobs=N_JOBS).fit(V).kneighbors(V)
    idx = idx[:, 1:]
    for b in BANDS:
        e = F[f'eres_{b}']
        add('local', f'loc_eres_{b}', e - np.median(e[idx], 1))
    s = F['size_res_raw']
    sn = s[idx]
    add('local', 'loc_size_q20', s - np.quantile(sn, 0.2, axis=1))
    add('local', 'loc_size_q50', s - np.median(sn, 1))
    point_like = (s < np.quantile(s, 0.35))[idx]
    for a, b in ADJ:
        c = F[f'{a}-{b}']
        med = np.nanmedian(np.where(point_like, c[idx], np.nan), 1)
        med = np.where(np.isnan(med), np.nanmedian(c), med)
        add('local', f'nb_col_{a}-{b}', med)
        add('local', f'loc_col_{a}-{b}', c - med)

    # 'dens': how crowded each object's neighbourhood is in colour space (train+test, no labels).
    # Stars crowd onto the stellar locus, quasars sit in sparse regions.
    # Experiment 31 Oct: LightGBM -0.0004 on average over 3 seeds.
    spaces = {'col4': ['u-g', 'g-r', 'r-i', 'i-z'], 'col4_shape': ['u-g', 'g-r', 'r-i', 'i-z', 'size_res'],
              'col4_mag': ['u-g', 'g-r', 'r-i', 'i-z', 'r']}
    for name, cols in spaces.items():
        Z = np.stack([F[c] for c in cols], 1).astype(np.float64)
        Z = (Z - Z.mean(0)) / Z.std(0)
        d, _ = NearestNeighbors(n_neighbors=51, n_jobs=N_JOBS).fit(Z).kneighbors(Z)
        for k in (10, 50):
            add('dens', f'dens_{name}_k{k}', np.log(d[:, k] + 1e-6))

    # 'nmix': noise-aware colour mixture (from the 31 Oct review; label-free, train+test).
    # An 8-component Gaussian mixture is fitted on the 4 adjacent colours of low-noise objects; each
    # object is then scored against every component WIDENED by its own colour-noise covariance
    # (adjacent colours share a band, hence the off-diagonal -err^2 terms). Features: the 8
    # responsibilities + total log-likelihood. Our 5-fold CV (tuned LightGBM, 3 seeds): -0.0014.
    from scipy.special import logsumexp, softmax
    from sklearn.mixture import GaussianMixture
    C = np.stack([F[f'{a}-{b}'] for a, b in ADJ], 1).astype(np.float64)
    e2 = np.stack([df[f'err_{b}'].values.astype(np.float64) ** 2 for b in BANDS], 1)
    cov = np.zeros((len(C), 4, 4))
    for i in range(4):
        cov[:, i, i] = e2[:, i] + e2[:, i + 1]
    for i in range(3):
        cov[:, i, i + 1] = cov[:, i + 1, i] = -e2[:, i + 1]
    clean = np.sqrt(np.stack([cov[:, i, i] for i in range(4)], 1)).max(1) < 0.12
    gm = GaussianMixture(8, covariance_type='full', reg_covar=1e-4, random_state=0, max_iter=200).fit(C[clean])
    R, LL = np.zeros((len(C), 8)), np.zeros(len(C))
    for a0 in range(0, len(C), 40000):
        sl = slice(a0, a0 + 40000)
        L = []
        for w, mu, S in zip(gm.weights_, gm.means_, gm.covariances_):
            Lc = np.linalg.cholesky(S[None] + cov[sl])
            z = np.linalg.solve(Lc, (C[sl] - mu)[..., None])[..., 0]
            L.append(np.log(w) - 0.5 * (z ** 2).sum(1) - np.log(np.diagonal(Lc, axis1=1, axis2=2)).sum(1))
        L = np.stack(L, 1)
        R[sl], LL[sl] = softmax(L, 1), logsumexp(L, 1)
    for k in range(8):
        add('nmix', f'nmix_r{k}', R[:, k])
    add('nmix', 'nmix_ll', LL)

    # 'spec' (V8, from the 1 Nov audit; untested gamble, trees only): spectral curvature, the SDSS
    # perpendicular colour, Lyman-break dropout, cross-band error ratios, error-residual spread and
    # a local seeing floor (15th percentile of log size among the 50 nearest sky neighbours).
    ug, gr, ri, iz = (F[f'{a}-{b}'] for a, b in ADJ)
    add('spec', 'curv_ugr', ug - gr)
    add('spec', 'curv_gri', gr - ri)
    add('spec', 'curv_riz', ri - iz)
    add('spec', 'd_perp', ri - gr / 4 - 0.18)
    add('spec', 'lyman_drop', ug - 2 * gr)
    add('spec', 'snr_ug_ratio', np.log(df['err_u'].values / df['err_g'].values))
    add('spec', 'snr_iz_ratio', np.log(df['err_z'].values / df['err_i'].values))
    add('spec', 'eres_std', E.std(1))
    add('spec', 'eres_z_minus_g', E[:, 4] - E[:, 1])
    add('spec', 'loc_raw_psf_diff', ls - np.quantile(ls[idx[:, :50]], 0.15, axis=1))

    X = pd.DataFrame(F)
    return X.iloc[:n_tr].reset_index(drop=True), X.iloc[n_tr:].reset_index(drop=True), groups

In [ ]:
%%writefile src/prep.py
"""Step 1: fixed folds, label-free features, and out-of-fold 'learned' features:
  photoz    - redshift predicted from photometry (allowed: confirmed by the coordinator; the real
              redshift is only ever a training target, never an input)
  knn_sky   - class mix of the nearest labelled objects on the sky (SDSS plates target one kind of
              object, so neighbours are informative)
  knn_color - class mix of the nearest labelled objects in colour/shape space
Train rows get out-of-fold values; test rows get the average over the 5 fold models, so both see
the same amount of labelled data."""
import json
import os

import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors

import features
from common import (CLASSES, FOLD_SEED, N_FOLDS, N_JOBS, SMOKE, WORK, Progress, labels, load_raw, log)

PROG = None   # set in main(): one progress counter across all 20 fold-steps

Z_BINS = [-1, 0.005, 0.3, 0.7, 1.5, 2.3, 3.0, 99]       # star | galaxy-ish | ... | high-z quasar


def oof_lgb(X, Xt, target, fold, params, rounds):
    """Out-of-fold predictions for train, fold-averaged predictions for test."""
    multi = params['objective'] == 'multiclass'
    k = params.get('num_class', 1)
    oof = np.zeros((len(X), k)) if multi else np.zeros(len(X))
    tst = np.zeros((len(Xt), k)) if multi else np.zeros(len(Xt))
    for f in range(N_FOLDS):
        a, b = fold != f, fold == f
        m = lgb.train(params, lgb.Dataset(X[a], target[a]), rounds,
                      valid_sets=[lgb.Dataset(X[b], target[b])],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict(X[b], num_iteration=m.best_iteration)
        tst += m.predict(Xt, num_iteration=m.best_iteration) / N_FOLDS
        PROG.step(f'{params["objective"]} fold {f} (best_iter {m.best_iteration})')
    return oof, tst


def oof_knn(Z, Zt, y, fold, ks, prefix):
    """Class fractions among the k nearest labelled neighbours, plus distance to the k-th."""
    kmax = max(ks)
    out = {f'{prefix}_k{k}_{c}': np.zeros(len(Z)) for k in ks for c in CLASSES}
    out.update({f'{prefix}_d{k}': np.zeros(len(Z)) for k in ks})
    outt = {n: np.zeros(len(Zt)) for n in out}
    for f in range(N_FOLDS):
        a, b = fold != f, fold == f
        nn = NearestNeighbors(n_neighbors=kmax, n_jobs=N_JOBS).fit(Z[a])
        ya = y[a]
        for rows, dst, w in ((b, out, None), (slice(None), outt, 1.0 / N_FOLDS)):
            q = Z[rows] if dst is out else Zt
            d, idx = nn.kneighbors(q)
            lab = ya[idx]
            for k in ks:
                for ci, c in enumerate(CLASSES):
                    v = (lab[:, :k] == ci).mean(1)
                    if w is None:
                        dst[f'{prefix}_k{k}_{c}'][rows] = v
                    else:
                        dst[f'{prefix}_k{k}_{c}'] += w * v
                v = np.log(d[:, k - 1] + 1e-9)
                if w is None:
                    dst[f'{prefix}_d{k}'][rows] = v
                else:
                    dst[f'{prefix}_d{k}'] += w * v
        PROG.step(f'{prefix} neighbours fold {f}')
    return out, outt


def main():
    global PROG
    PROG = Progress('prep', 4 * N_FOLDS)
    tr, te = load_raw()
    y = labels(tr)
    fold = np.zeros(len(tr), dtype=np.int64)
    for f, (_, b) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=FOLD_SEED).split(tr, y)):
        fold[b] = f
    np.save(WORK / 'y.npy', y)
    np.save(WORK / 'fold.npy', fold)
    np.save(WORK / 'test_ids.npy', te['object_id'].values)
    np.save(WORK / 'redshift.npy', tr['redshift'].values)   # training target only (MLP aux head)

    log('label-free features')
    Xtr, Xte, groups = features.build(tr, te)
    base = [c for g in ('raw', 'color', 'flux', 'morph', 'errres', 'sky') for c in groups[g]]
    A, At = Xtr[base].values, Xte[base].values
    rounds = 200 if SMOKE else 5000

    # V8: the old out-of-fold learned features are unused by the current models (they use the clean
    # nested build instead); CS_PREP_LEARNED=0 skips them and saves ~3-4 min on Kaggle.
    if os.environ.get('CS_PREP_LEARNED', '1') == '1':
        log('photo-z regression (target: log1p redshift)')
        zt = np.log1p(np.clip(tr['redshift'].values, 0, None))
        # lr 0.1 / 63 leaves: ~2x faster than lr 0.05 / 127 leaves, and photo-z barely changes the score
        p = dict(objective='regression', learning_rate=0.1, num_leaves=63, min_child_samples=20,
                 feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                 verbose=-1, num_threads=N_JOBS, seed=0)
        pz, pzt = oof_lgb(A, At, zt, fold, p, rounds)
        log(f'  photo-z RMSE (log1p z): {np.sqrt(np.mean((pz - zt) ** 2)):.4f}')
        Xtr['pz'], Xte['pz'] = pz, pzt
        groups['photoz'] = ['pz']

        log('photo-z bins (multiclass over redshift ranges)')
        zb = np.digitize(tr['redshift'].values, Z_BINS[1:-1])
        p = dict(p, objective='multiclass', num_class=len(Z_BINS) - 1)
        pb, pbt = oof_lgb(A, At, zb, fold, p, rounds)
        for k in range(pb.shape[1]):
            Xtr[f'pzb_{k}'], Xte[f'pzb_{k}'] = pb[:, k], pbt[:, k]
            groups['photoz'].append(f'pzb_{k}')
        Xtr['pzb_mean'] = pb @ np.arange(pb.shape[1])
        Xte['pzb_mean'] = pbt @ np.arange(pb.shape[1])
        groups['photoz'].append('pzb_mean')

        log('sky neighbours')
        V, Vt = features.sky_vectors(tr), features.sky_vectors(te)
        o, ot = oof_knn(V, Vt, y, fold, [10, 30, 100], 'sky')
        for n in o:
            Xtr[n], Xte[n] = o[n], ot[n]
        groups['knn_sky'] = list(o)

        log('colour-space neighbours')
        cc = ['u-g', 'g-r', 'r-i', 'i-z', 'size_res', 'r']
        mu, sd = Xtr[cc].mean(), Xtr[cc].std()
        Z, Zt = ((Xtr[cc] - mu) / sd).values, ((Xte[cc] - mu) / sd).values
        Z[:, -1] *= 0.5
        Zt[:, -1] *= 0.5                                         # brightness counts half as much as colour
        o, ot = oof_knn(Z, Zt, y, fold, [25, 100], 'cknn')
        for n in o:
            Xtr[n], Xte[n] = o[n], ot[n]
        groups['knn_color'] = list(o)

    Xtr = Xtr.astype(np.float32)
    Xte = Xte.astype(np.float32)
    Xtr.to_pickle(WORK / 'feat_train.pkl')
    Xte.to_pickle(WORK / 'feat_test.pkl')
    (WORK / 'feature_groups.json').write_text(json.dumps(groups, indent=1))
    log(f'done: train {Xtr.shape}, test {Xte.shape}, groups ' +
        ', '.join(f'{g}={len(c)}' for g, c in groups.items()))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/nested.py
"""Nested ('clean') learned features: photo-z + neighbour class mix, rebuilt separately for every outer
fold f using ONLY the labels of that fold's training rows.

Why (31 Oct leak test, holdout = fold 0, same LightGBM): learned features built by prep.py are
out-of-fold over the 5 folds, so a training row's feature was computed with the validation rows'
labels. Rebuilt cleanly, the all-features model scored 0.1556 vs 0.1583 (standard) and 0.1582
(no learned features). The contamination was hiding a real signal.

For outer fold f (train rows R = folds != f):
  R rows    -> out-of-fold over R's own 4 folds (inner CV)
  f rows    -> average of the 4 inner models (like test)
  test rows -> average of the 4 inner models
Saved to work/nested/f{f}.npz (train / val / test arrays + column names). Used by
train_gbdt.py --nested and train_mlp.py --nested.

  python src/nested.py"""
import os

import numpy as np

import features
import prep
from common import N_FOLDS, N_JOBS, SMOKE, WORK, Progress, load_raw, load_work, log

OUT = WORK / os.environ.get('CS_NESTED_DIR', 'nested')
# 31 Oct: all the gain comes from the photo-z features (pz + 7 redshift-range probabilities);
# the neighbour class mix added nothing, so it is off by default (CS_NESTED_KNN=1 turns it on).
WITH_KNN = os.environ.get('CS_NESTED_KNN', '0') == '1'
FAST = os.environ.get('CS_NESTED_FAST', '1') == '1'     # lr 0.2 / 31 leaves / 127 bins: ~3x faster


def build_fold(f, tr, te, y, fold, raw_tr, raw_te, groups):
    R, V = fold != f, fold == f
    inner = fold[R] - (fold[R] > f)                          # R's folds renumbered 0..3
    prep.N_FOLDS = N_FOLDS - 1
    lab_free = [c for g in ('raw', 'color', 'flux', 'morph', 'errres', 'sky') for c in groups[g]]
    A = tr.loc[R, lab_free].values
    O = np.vstack([tr.loc[V, lab_free].values, te[lab_free].values])   # val rows + test rows
    nV = V.sum()
    cols, trainF, otherF = [], [], []

    def put(name, a, b):
        cols.append(name)
        trainF.append(a)
        otherF.append(b)

    rounds = 200 if SMOKE else 5000
    zt = np.log1p(np.clip(raw_tr['redshift'].values, 0, None))[R]
    p = dict(objective='regression', learning_rate=0.1, num_leaves=63, min_child_samples=20, feature_fraction=0.8,
             bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=N_JOBS, seed=0)
    if FAST:
        p.update(learning_rate=0.2, num_leaves=31, max_bin=127)
    a, b = prep.oof_lgb(A, O, zt, inner, p, rounds)
    put('pz', a, b)
    zb = np.digitize(raw_tr['redshift'].values, prep.Z_BINS[1:-1])[R]
    a, b = prep.oof_lgb(A, O, zb, inner, dict(p, objective='multiclass', num_class=len(prep.Z_BINS) - 1), rounds)
    for k in range(a.shape[1]):
        put(f'pzb_{k}', a[:, k], b[:, k])
    put('pzb_mean', a @ np.arange(a.shape[1]), b @ np.arange(a.shape[1]))

    if not WITH_KNN:
        return save(f, cols, trainF, otherF, nV)
    Vall = features.sky_vectors(raw_tr)
    Vo = np.vstack([Vall[V], features.sky_vectors(raw_te)])
    o, oo = prep.oof_knn(Vall[R], Vo, y[R], inner, [10, 30, 100], 'sky')
    for n in o:
        put(n, o[n], oo[n])
    cc = ['u-g', 'g-r', 'r-i', 'i-z', 'size_res', 'r']
    mu, sd = tr.loc[R, cc].mean(), tr.loc[R, cc].std()
    Z = ((tr[cc] - mu) / sd).values
    Zt = ((te[cc] - mu) / sd).values
    Z[:, -1] *= 0.5
    Zt[:, -1] *= 0.5
    o, oo = prep.oof_knn(Z[R], np.vstack([Z[V], Zt]), y[R], inner, [25, 100], 'cknn')
    for n in o:
        put(n, o[n], oo[n])

    return save(f, cols, trainF, otherF, nV)


def save(f, cols, trainF, otherF, nV):
    T = np.stack(trainF, 1).astype(np.float32)
    O = np.stack(otherF, 1).astype(np.float32)
    np.savez_compressed(OUT / f'f{f}.npz', train=T, val=O[:nV], test=O[nV:], cols=np.array(cols))
    prep.N_FOLDS = N_FOLDS
    return cols


def load_fold(f):
    d = np.load(OUT / f'f{f}.npz')
    return d['train'], d['val'], d['test'], list(d['cols'])


def main():
    tr, te, y, fold, groups = load_work()
    raw_tr, raw_te = load_raw()
    OUT.mkdir(exist_ok=True)
    prep.PROG = Progress('nested learned features', N_FOLDS * (4 if WITH_KNN else 2) * (N_FOLDS - 1))
    for f in range(N_FOLDS):
        cols = build_fold(f, tr, te, y, fold, raw_tr, raw_te, groups)
        log(f'outer fold {f}: {len(cols)} learned features saved')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/train_gbdt.py
"""Train LightGBM / XGBoost / CatBoost on the fixed folds; save OOF + test predictions.

  python src/train_gbdt.py --model lgb --features all --seeds 0,1,2 --name lgb_all
Tuned params from work/params/<model>.json (written by tune.py) are used when present."""
import argparse
import json

import numpy as np

from common import (N_FOLDS, N_JOBS, PARAMS, SMOKE, Progress, load_work, log, mlogloss, save_preds,
                    select, split_folds)

DEFAULTS = {
    'lgb': dict(objective='multiclass', num_class=3, learning_rate=0.03, num_leaves=63,
                min_child_samples=30, feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1,
                lambda_l1=0.0, lambda_l2=1.0, max_bin=255, verbose=-1),
    'xgb': dict(objective='multi:softprob', num_class=3, eval_metric='mlogloss', learning_rate=0.03,
                max_depth=7, min_child_weight=2, subsample=0.8, colsample_bytree=0.6,
                reg_lambda=1.0, reg_alpha=0.0, tree_method='hist', max_bin=256),
    'cat': dict(loss_function='MultiClass', learning_rate=0.05, depth=8, l2_leaf_reg=3.0,
                border_count=254, random_strength=1.0, bootstrap_type='Bernoulli', subsample=0.8),
}


def get_params(model, lr=None, tuned=True):
    p = dict(DEFAULTS[model])
    f = PARAMS / f'{model}.json'
    if tuned and f.exists():
        p.update(json.loads(f.read_text())['params'])
        log(f'using tuned params from {f}')
    if lr:
        p['learning_rate'] = lr
    return p


def fit_predict(model, p, Xa, ya, Xb, yb, Xt, seed, device, rounds, es=200):
    """Train on (Xa, ya) with early stopping on (Xb, yb); return (pred_b, pred_test, best_iter)."""
    if model == 'lgb':
        import lightgbm as lgb
        q = dict(p, seed=seed, num_threads=N_JOBS)
        m = lgb.train(q, lgb.Dataset(Xa, ya), rounds, valid_sets=[lgb.Dataset(Xb, yb)],
                      callbacks=[lgb.early_stopping(es, verbose=False)])
        it = m.best_iteration
        return m.predict(Xb, num_iteration=it), m.predict(Xt, num_iteration=it), it
    if model == 'xgb':
        import xgboost as xgb
        q = dict(p, seed=seed, nthread=N_JOBS)
        if device == 'gpu':
            q['device'] = 'cuda'
        da, db, dt = xgb.DMatrix(Xa, ya), xgb.DMatrix(Xb, yb), xgb.DMatrix(Xt)
        m = xgb.train(q, da, rounds, evals=[(db, 'valid')], early_stopping_rounds=es, verbose_eval=False)
        r = (0, m.best_iteration + 1)
        return m.predict(db, iteration_range=r), m.predict(dt, iteration_range=r), m.best_iteration + 1
    if model == 'cat':
        from catboost import CatBoostClassifier
        q = dict(p, iterations=rounds, random_seed=seed, od_type='Iter', od_wait=es, verbose=0,
                 use_best_model=True, allow_writing_files=False)
        if device == 'gpu':
            q.update(task_type='GPU', devices='0')
        else:
            q['thread_count'] = N_JOBS
        m = CatBoostClassifier(**q)
        m.fit(Xa, ya, eval_set=(Xb, yb))
        return m.predict_proba(Xb), m.predict_proba(Xt), m.get_best_iteration()
    raise ValueError(model)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True, choices=['lgb', 'xgb', 'cat'])
    ap.add_argument('--features', default='all')
    ap.add_argument('--seeds', default='0')
    ap.add_argument('--lr', type=float, default=None)
    ap.add_argument('--rounds', type=int, default=30000)
    ap.add_argument('--device', default='cpu', choices=['cpu', 'gpu'])
    ap.add_argument('--no-tuned', action='store_true')
    ap.add_argument('--nfolds', type=int, default=N_FOLDS, help='5, 10 or 20 (more folds = more training rows per model)')
    ap.add_argument('--nested', action='store_true',
                    help='add the clean per-fold photo-z features from src/nested.py (5 or 10 folds)')
    ap.add_argument('--set', action='append', default=[], metavar='KEY=VALUE',
                    help='override one model param, e.g. --set extra_trees=true (repeatable)')
    ap.add_argument('--name', default=None)
    a = ap.parse_args()
    if SMOKE:
        a.rounds = 150

    tr, te, y, fold, groups = load_work()
    fold5 = fold.copy()
    fold = split_folds(fold, y, a.nfolds)
    nf = a.nfolds
    cols = select(groups, a.features)
    X, Xt = tr[cols].values, te[cols].values
    p = get_params(a.model, a.lr, not a.no_tuned)
    for kv in a.set:
        k, v = kv.split('=', 1)
        p[k] = json.loads(v) if v[:1] in '0123456789-[{' or v in ('true', 'false') else v
    name = a.name or f'{a.model}_{a.features.replace(",", "_")}'
    seeds = [int(s) for s in a.seeds.split(',')]
    log(f'{name}: {len(cols)} features, seeds {seeds}, params {p}')

    oof = np.zeros((len(X), 3))
    tst = np.zeros((len(Xt), 3))
    iters = []
    prog = Progress(name, len(seeds) * nf)
    for s in seeds:
        o = np.zeros((len(X), 3))
        for f in range(nf):
            a_, b_ = fold != f, fold == f
            Xa, Xb, Xtf = X[a_], X[b_], Xt
            if a.nested:
                # clean photo-z features of the 5-fold group g that holds this validation fold: built
                # without any of group g's labels, so they are clean for both of g's 10-fold halves
                from nested import load_fold
                g = f // (nf // N_FOLDS)          # 5-fold group holding validation fold f
                T, V, Lt, _ = load_fold(g)
                L = np.zeros((len(X), T.shape[1]), dtype=np.float32)
                L[fold5 != g], L[fold5 == g] = T, V
                Xa, Xb, Xtf = np.hstack([Xa, L[a_]]), np.hstack([Xb, L[b_]]), np.hstack([Xt, Lt])
            pb, pt, it = fit_predict(a.model, p, Xa, y[a_], Xb, y[b_], Xtf, s, a.device, a.rounds)
            o[b_] = pb
            tst += pt / (nf * len(seeds))
            iters.append(int(it))
            prog.step(f'seed {s} fold {f}: {mlogloss(y[b_], pb):.5f} (iter {it})')
        log(f'  seed {s}: CV {mlogloss(y, o):.5f}')
        oof += o / len(seeds)
    save_preds(name, oof, tst, y, dict(model=a.model, features=a.features, n_features=len(cols),
                                       nested=a.nested, seeds=a.seeds, nfolds=nf, iters=iters, params=p))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/train_mlp.py
"""PyTorch MLP on quantile-normalised features, with an auxiliary redshift head (redshift is only a
training target here; at test time the inputs are photometry-derived features only).

  python src/train_mlp.py --features all --seeds 0,1,2 --name mlp_all"""
import argparse
import copy

import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import QuantileTransformer

from common import (N_FOLDS, N_JOBS, SMOKE, WORK, Progress, load_work, log, mlogloss, save_preds,
                    select, split_folds)


class PeriodicEmb(nn.Module):
    """Periodic embeddings for numbers (Gorishniy et al. 2022, 'PLR'): each feature x becomes
    ReLU(W_f [sin(2πc_f x), cos(2πc_f x)]) with learned frequencies c_f, then all are concatenated."""

    def __init__(self, n_feat, n_freq, d_out, sigma):
        super().__init__()
        self.c = nn.Parameter(torch.randn(n_feat, n_freq) * sigma)
        self.w = nn.Parameter(torch.randn(n_feat, 2 * n_freq, d_out) / (2 * n_freq) ** 0.5)
        self.b = nn.Parameter(torch.zeros(n_feat, d_out))

    def forward(self, x):
        v = 2 * np.pi * x.unsqueeze(-1) * self.c                      # (B, F, K)
        z = torch.cat([torch.sin(v), torch.cos(v)], -1)                # (B, F, 2K)
        return torch.relu(torch.einsum('bfk,fkd->bfd', z, self.w) + self.b).flatten(1)


class MLP(nn.Module):
    def __init__(self, d_in, hidden, layers, dropout, emb=None):
        super().__init__()
        self.emb = None
        if emb and emb[0] > 0:                                         # emb = (n_freq, d_out, sigma)
            self.emb = PeriodicEmb(d_in, *emb)
            d_in = d_in * emb[1]
        blocks, d = [], d_in
        for _ in range(layers):
            blocks += [nn.Linear(d, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
            d = hidden
        self.body = nn.Sequential(*blocks)
        self.cls = nn.Linear(d, 3)
        self.z = nn.Linear(d, 1)

    def forward(self, x):
        if self.emb is not None:
            x = self.emb(x)
        h = self.body(x)
        return self.cls(h), self.z(h).squeeze(1)


def predict(model, X, dev, bs=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(torch.softmax(model(X[i:i + bs].to(dev))[0], 1).cpu())
    return torch.cat(out).numpy().astype(np.float64)


def train_fold(Xa, ya, za, Xb, yb, Xt, a, seed, dev):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = MLP(Xa.shape[1], a.hidden, a.layers, a.dropout,
                (a.emb_freq, a.emb_dim, a.emb_sigma)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    steps = len(Xa) // a.bs                    # drop the last partial batch (BatchNorm needs > 1 row)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=a.epochs * steps,
                                                pct_start=0.1)
    ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()
    Xa_d, ya_d, za_d = Xa.to(dev), ya.to(dev), za.to(dev)
    g = torch.Generator(device='cpu').manual_seed(seed)
    ema = None
    if a.ema > 0:                       # evaluate an exponential moving average of the weights
        from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
        ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(a.ema), use_buffers=True)
    best, best_state, bad = 1e9, None, 0
    for ep in range(a.epochs):
        model.train()
        perm = torch.randperm(len(Xa), generator=g).to(dev)
        for i in range(steps):
            idx = perm[i * a.bs:(i + 1) * a.bs]
            xb = Xa_d[idx]
            if a.noise > 0:                 # small Gaussian jitter on the (quantile-normal) inputs
                xb = xb + a.noise * torch.randn_like(xb)
            logit, zp = model(xb)
            loss = ce(logit, ya_d[idx]) + a.aux * mse(zp, za_d[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            if ema is not None:
                ema.update_parameters(model)
        evm = ema.module if ema is not None else model
        vl = mlogloss(yb.numpy(), predict(evm, Xb, dev))
        if vl < best - 1e-5:
            best, best_state, bad = vl, copy.deepcopy(evm.state_dict()), 0
        else:
            bad += 1
        if bad >= a.patience:
            break
    model.load_state_dict(best_state)
    return predict(model, Xb, dev), predict(model, Xt, dev), best, ep + 1


def parser():
    ap = argparse.ArgumentParser()
    ap.add_argument('--features', default='all')
    ap.add_argument('--seeds', default='0')
    ap.add_argument('--hidden', type=int, default=512)
    ap.add_argument('--layers', type=int, default=3)
    ap.add_argument('--dropout', type=float, default=0.15)
    ap.add_argument('--lr', type=float, default=2e-3)
    ap.add_argument('--wd', type=float, default=1e-4)
    ap.add_argument('--bs', type=int, default=512)
    ap.add_argument('--epochs', type=int, default=40)
    ap.add_argument('--patience', type=int, default=8)
    ap.add_argument('--aux', type=float, default=0.2, help='weight of the redshift loss (0 = off)')
    ap.add_argument('--emb-freq', type=int, default=0, help='periodic embeddings: frequencies (0 = off)')
    ap.add_argument('--emb-dim', type=int, default=16)
    ap.add_argument('--emb-sigma', type=float, default=0.3)
    ap.add_argument('--scaler', default='quantile', choices=['quantile', 'robust'])
    ap.add_argument('--nfolds', type=int, default=N_FOLDS, help='5, 10 or 20 (more folds = more training rows per model)')
    ap.add_argument('--nested', action='store_true',
                    help='add the per-fold clean photo-z features from src/nested.py (5 folds only)')
    ap.add_argument('--noise', type=float, default=0.0, help='Gaussian input noise during training (0 = off)')
    ap.add_argument('--ema', type=float, default=0.0, help='EMA decay for the weights, e.g. 0.999 (0 = off)')
    ap.add_argument('--only-fold', type=int, default=None, help='quick experiment: train one fold, save nothing')
    ap.add_argument('--name', default=None)
    return ap


def load_inputs(spec, scaler='quantile'):
    """Normalised features (fit on train+test, label-free), labels, redshift target, folds.
    scaler='quantile': rank -> normal (default). scaler='robust': clip to the 0.5-99.5 percentiles,
    then (x - median) / IQR; keeps distances between values, so the net learns different patterns."""
    tr, te, y, fold, groups = load_work()
    cols = select(groups, spec)
    allX = np.vstack([tr[cols].values, te[cols].values]).astype(np.float64)
    allX = np.nan_to_num(allX, nan=0.0, posinf=1e6, neginf=-1e6)
    if scaler == 'robust':
        lo, hi = np.percentile(allX, [0.5, 99.5], axis=0)
        allX = np.clip(allX, lo, hi)
        q25, q50, q75 = np.percentile(allX, [25, 50, 75], axis=0)
        allX = ((allX - q50) / np.maximum(q75 - q25, 1e-9)).astype(np.float32)
    else:
        qt = QuantileTransformer(n_quantiles=2000, output_distribution='normal', subsample=10 ** 9,
                                 random_state=0).fit(allX)
        allX = qt.transform(allX).astype(np.float32)
    z = np.log1p(np.clip(np.load(WORK / 'redshift.npy'), 0, None))
    z = (z - z.mean()) / z.std()
    return (torch.from_numpy(allX[:len(tr)]), torch.from_numpy(allX[len(tr):]), y, torch.from_numpy(y),
            torch.from_numpy(z.astype(np.float32)), fold, cols)


def main(argv=None):
    a = parser().parse_args(argv)
    if SMOKE:
        a.epochs = 3
    torch.set_num_threads(N_JOBS)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    X, Xt, y, yt, zt, fold, cols = load_inputs(a.features, a.scaler)
    assert not (a.nested and a.nfolds != N_FOLDS), '--nested needs the 5 original folds'
    fold = split_folds(fold, y, a.nfolds)
    nf = a.nfolds

    def fold_inputs(f, A, B):
        """Inputs for fold f; with --nested, append that fold's clean photo-z features, quantile-
        normalised with a transform fitted on the fold's training rows only."""
        if not a.nested:
            return X[A], X[B], Xt
        from nested import load_fold
        La, Lb, Lt, _ = load_fold(f)
        q = QuantileTransformer(n_quantiles=2000, output_distribution='normal', subsample=10 ** 9,
                                random_state=0).fit(La)
        t = lambda M: torch.from_numpy(q.transform(M).astype(np.float32))  # noqa: E731
        return torch.cat([X[A], t(La)], 1), torch.cat([X[B], t(Lb)], 1), torch.cat([Xt, t(Lt)], 1)
    name = a.name or f'mlp_{a.features.replace(",", "_")}'
    seeds = [int(s) for s in a.seeds.split(',')]
    log(f'{name}: {len(cols)} features on {dev}, seeds {seeds}, {vars(a)}')

    if a.only_fold is not None:
        f = a.only_fold
        A, B = torch.from_numpy(fold != f), torch.from_numpy(fold == f)
        Xa, Xb, Xtf = fold_inputs(f, A, B)
        vl = np.mean([train_fold(Xa, yt[A], zt[A], Xb, yt[B], Xtf[:2], a, 1000 * s + f, dev)[2]
                      for s in seeds])
        log(f'RESULT {name} fold {f}: {vl:.5f}')
        return vl

    oof = np.zeros((len(X), 3))
    tst = np.zeros((len(Xt), 3))
    prog = Progress(name, len(seeds) * nf)
    for s in seeds:
        o = np.zeros((len(X), 3))
        for f in range(nf):
            A = torch.from_numpy(fold != f)
            B = torch.from_numpy(fold == f)
            Xa, Xb, Xtf = fold_inputs(f, A, B)
            pb, pt, vl, ep = train_fold(Xa, yt[A], zt[A], Xb, yt[B], Xtf, a, 1000 * s + f, dev)
            o[B.numpy()] = pb
            tst += pt / (nf * len(seeds))
            prog.step(f'seed {s} fold {f}: {vl:.5f} ({ep} epochs)')
        log(f'  seed {s}: CV {mlogloss(y, o):.5f}')
        oof += o / len(seeds)
    return save_preds(name, oof, tst, y, dict(model='mlp', features=a.features, n_features=len(cols),
                                              nested=a.nested, seeds=a.seeds, nfolds=nf, args=vars(a)))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/ensemble.py
"""Blend saved model predictions, calibrate, and write a Kaggle submission.

  python src/ensemble.py                       # every model in work/preds
  python src/ensemble.py --models lgb_all,mlp_all --tag v2

Three blenders are scored with nested CV on the fixed folds (weights fit on 4 folds, scored on the
5th), and the best one is used:
  linear  - weighted average of probabilities (+ temperature scaling)
  geo     - weighted average of log-probabilities + per-class bias (calibration built in)
  stack   - multinomial logistic regression on all models' log-probabilities"""
import argparse
import json
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import log_softmax, softmax
from sklearn.linear_model import LogisticRegression

from common import CLASSES, N_FOLDS, PREDS, SMOKE, SUBS, WORK, load_work, log, mlogloss


def fit_linear(P, y, E=None):
    """P: (models, n, 3). Softmax-parametrised weights, then one temperature."""
    M = len(P)

    def blend(w, Q):
        return np.tensordot(softmax(w), Q, 1)

    w = minimize(lambda w: mlogloss(y, blend(w, P)), np.zeros(M), method='L-BFGS-B').x
    lb = np.log(np.clip(blend(w, P), 1e-15, 1))
    T = minimize(lambda t: mlogloss(y, softmax(lb / t[0], 1)), [1.0], bounds=[(0.5, 3)],
                 method='L-BFGS-B').x[0]

    def apply(Q, Eq=None):
        return softmax(np.log(np.clip(blend(w, Q), 1e-15, 1)) / T, 1)
    return apply, dict(weights=softmax(w).round(4).tolist(), T=round(float(T), 4))


def fit_geo(P, y, E=None):
    L = np.log(np.clip(P, 1e-15, 1))
    M = len(P)

    def blend(th, Q):
        return log_softmax(np.tensordot(th[:M], Q, 1) + th[M:], 1)

    Y = np.eye(3)[y]
    n = len(y)

    def nll(th):                       # loss and its exact gradient (much faster than numerical)
        lp = blend(th, L)
        G = (np.exp(lp) - Y) / n                     # d loss / d logits
        return -(lp * Y).sum() / n, np.r_[np.einsum('mnc,nc->m', L, G), G.sum(0)]

    th = minimize(nll, np.r_[np.ones(M) / M, np.zeros(3)], jac=True, method='L-BFGS-B').x

    def apply(Q, Eq=None):
        return np.exp(blend(th, np.log(np.clip(Q, 1e-15, 1))))
    return apply, dict(weights=th[:M].round(4).tolist(), bias=th[M:].round(4).tolist())


def fit_geo_bounded(P, y, E=None, l2=1e-4):
    """V8 (from the 1 Nov audit): like 'geo' but every model weight is kept in [0, 1.2] and gently
    pulled toward equal weights, so correlated models can't get offsetting negative weights.
    Offered as an extra candidate; nested CV decides whether it is used."""
    L = np.log(np.clip(P, 1e-15, 1))
    M = len(P)
    Y = np.eye(3)[y]
    n = len(y)
    u = np.ones(M) / M

    def blend(th, Q):
        return log_softmax(np.tensordot(th[:M], Q, 1) + th[M:], 1)

    def nll(th):
        lp = blend(th, L)
        G = (np.exp(lp) - Y) / n
        return (-(lp * Y).sum() / n + 0.5 * l2 * ((th[:M] - u) ** 2).sum(),
                np.r_[np.einsum('mnc,nc->m', L, G) + l2 * (th[:M] - u), G.sum(0)])

    th = minimize(nll, np.r_[u, np.zeros(3)], jac=True, method='L-BFGS-B',
                  bounds=[(0.0, 1.2)] * M + [(-0.5, 0.5)] * 3).x

    def apply(Q, Eq=None):
        return np.exp(blend(th, np.log(np.clip(Q, 1e-15, 1))))
    return apply, dict(weights=th[:M].round(4).tolist(), bias=th[M:].round(4).tolist())


def fit_stack(P, y, E=None, C=1.0):
    def feats(Q):
        L = np.log(np.clip(Q, 1e-15, 1))
        return np.concatenate(list(L), 1)
    lr = LogisticRegression(C=C, max_iter=2000).fit(feats(P), y)

    def apply(Q, Eq=None):
        return lr.predict_proba(feats(Q))
    return apply, dict(C=C)


META_COLS = ['r', 'size_res', 'abs_b', 'u-g', 'g-r', 'r-i', 'eres_mean', 'loc_size_q20']


def fit_meta(P, y, E, residual=True):
    """Small LightGBM on every model's log-probabilities + a few key features, so the blend can
    trust different models for faint/bright, small/large, blue/red objects. With residual=True it
    starts from the 'geo' blend (init_score) and only learns corrections on top of it. Early
    stopping uses an inner 15% holdout of the rows it is given (never the rows it is scored on)."""
    import lightgbm as lgb
    from common import N_JOBS

    def feats(Q, Eq):
        return np.concatenate(list(np.log(np.clip(Q, 1e-15, 1))) + [Eq], 1)
    base_apply, _ = fit_geo(P, y) if residual else (None, None)

    def base(Q):
        return np.log(np.clip(base_apply(Q), 1e-15, 1)) if residual else None
    X, B = feats(P, E), base(P)
    prm = dict(objective='multiclass', num_class=3, learning_rate=0.02, num_leaves=15,
               min_child_samples=300, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
               lambda_l2=10.0, verbose=-1, num_threads=N_JOBS, seed=0)
    hold = np.random.default_rng(0).random(len(y)) < 0.15

    def ds(rows):
        return lgb.Dataset(X[rows], y[rows], init_score=None if B is None else B[rows])
    m = lgb.train(prm, ds(~hold), 3000, valid_sets=[ds(hold)], callbacks=[lgb.early_stopping(100, verbose=False)])
    it = max(1, int(m.best_iteration * 1.1))
    m = lgb.train(prm, ds(np.ones(len(y), bool)), it)

    def apply(Q, Eq=None):
        raw = m.predict(feats(Q, Eq), raw_score=True)
        return softmax(raw + base(Q), 1) if residual else softmax(raw, 1)
    return apply, dict(rounds=it, residual=residual)


def nested(fitter, P, y, fold, E):
    oof = np.zeros((P.shape[1], 3))
    for f in range(N_FOLDS):
        a, b = fold != f, fold == f
        apply, _ = fitter(P[:, a], y[a], E[a])
        oof[b] = apply(P[:, b], E[b])
    return oof


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', default='all')
    ap.add_argument('--exclude', default='')
    ap.add_argument('--tag', default=time.strftime('%m%d_%H%M'))
    ap.add_argument('--out', default=None, help='also write the submission here (Kaggle: submission.csv)')
    ap.add_argument('--no-meta', action='store_true', help='skip the LightGBM meta-blender')
    a = ap.parse_args()

    tr, te, y, fold, _ = load_work()
    names = sorted(p.stem for p in PREDS.glob('*.npz')) if a.models == 'all' else a.models.split(',')
    names = [n for n in names if n not in a.exclude.split(',')]
    P, Pt = [], []
    for n in names:
        d = np.load(PREDS / f'{n}.npz')
        P.append(d['oof'].astype(np.float64))
        Pt.append(d['test'].astype(np.float64))
    P, Pt = np.stack(P), np.stack(Pt)
    P /= P.sum(2, keepdims=True)
    Pt /= Pt.sum(2, keepdims=True)
    log('single models (CV log loss):')
    for n, p in sorted(zip(names, P), key=lambda t: mlogloss(y, t[1])):
        log(f'  {mlogloss(y, p):.5f}  {n}')

    mc = [c for c in META_COLS if c in tr.columns]
    E, Et = tr[mc].values.astype(np.float64), te[mc].values.astype(np.float64)
    fitters = {'linear': fit_linear, 'geo': fit_geo, 'geo_bounded': fit_geo_bounded}
    for C in (0.1, 1.0):
        fitters[f'stack_C{C}'] = lambda P_, y_, E_=None, C=C: fit_stack(P_, y_, E_, C)
    if not a.no_meta:
        fitters['meta_lgb'] = fit_meta
    scores, oofs = {}, {}
    for k, fn in fitters.items():
        oofs[k] = nested(fn, P, y, fold, E)
        scores[k] = mlogloss(y, oofs[k])
        log(f'blend {k:10s} nested-CV {scores[k]:.5f}')
    best = min(scores, key=scores.get)
    apply, info = fitters[best](P, y, E)
    oof, test = oofs[best], apply(Pt, Et)

    # clipping: pick the floor on nested OOF predictions
    def clip(p, e):
        p = np.clip(p, e, 1)
        return p / p.sum(1, keepdims=True)
    eps = min([0, 1e-6, 1e-5, 1e-4], key=lambda e: mlogloss(y, clip(oof, e) if e else oof))
    if eps:
        oof, test = clip(oof, eps), clip(test, eps)
    final = mlogloss(y, oof)
    log(f'chosen: {best} {info} | clip {eps} | nested-CV {final:.5f}')

    # where is the loss now?
    loss = -np.log(np.clip(oof[np.arange(len(y)), y], 1e-15, 1))
    df = pd.DataFrame(dict(cls=np.array(CLASSES)[y], r=tr['r'].values, loss=loss))
    df['r_bin'] = pd.cut(df.r, [0, 18, 19, 20, 21, 22, 40])
    rep = df.pivot_table(index='r_bin', columns='cls', values='loss', aggfunc='mean', observed=True)
    log('mean loss by r magnitude x true class:\n' + rep.round(3).to_string())
    log('share of total loss by class: ' +
        ', '.join(f'{c} {df.loss[df.cls == c].sum() / df.loss.sum():.3f}' for c in CLASSES))

    ids = np.load(WORK / 'test_ids.npy')
    sub = pd.DataFrame(test, columns=CLASSES)
    sub.insert(0, 'object_id', ids)
    assert not sub.isna().any().any() and np.allclose(sub[CLASSES].sum(1), 1, atol=1e-6)
    assert SMOKE or len(sub) == 31000, len(sub)
    path = SUBS / f'sub_{a.tag}_cv{final:.5f}.csv'
    sub.to_csv(path, index=False)
    if a.out:
        sub.to_csv(a.out, index=False)
        log(f'WROTE {a.out}')
    (WORK / 'ensembles').mkdir(exist_ok=True)
    np.savez_compressed(WORK / 'ensembles' / f'{a.tag}.npz', oof=oof, test=test)
    with open(WORK / 'ensembles.jsonl', 'a') as f:
        f.write(json.dumps(dict(tag=a.tag, models=names, method=best, info=info, clip=eps,
                                nested_cv=round(final, 6), all=scores, file=path.name)) + '\n')
    log(f'WROTE {path}')


if __name__ == '__main__':
    main()

## Run (live progress below; also saved to `progress.md`)

In [ ]:
# feature set chosen by src/ablate.py and params from src/tune.py (both run offline)
import pathlib
W = pathlib.Path(WD) / 'work'; (W / 'params').mkdir(parents=True, exist_ok=True)
(W / 'best_features.txt').write_text('raw,color,flux,morph,errres,sky,local,dens,nmix,spec')
PARAMS = {
 "lgb": {
  "model": "lgb",
  "features": "best",
  "tune_lr": 0.08,
  "score": 0.15519564375267847,
  "trial": 72,
  "iters": [
   140,
   132
  ],
  "params": {
   "num_leaves": 129,
   "min_child_samples": 6,
   "feature_fraction": 0.33566272615195697,
   "bagging_fraction": 0.9867093951206547,
   "lambda_l1": 3.0848736137035792e-06,
   "lambda_l2": 5.972915764461234,
   "min_gain_to_split": 0.3893637536883358,
   "path_smooth": 26.155304601473887
  }
 },
 "xgb": {
  "model": "xgb",
  "features": "best",
  "tune_lr": 0.08,
  "score": 0.1561363786458969,
  "trial": 27,
  "iters": [
   225,
   247
  ],
  "params": {
   "max_depth": 11,
   "min_child_weight": 10.569026150555239,
   "subsample": 0.8061110747783549,
   "colsample_bytree": 0.7402104823925528,
   "reg_lambda": 26.132280211577147,
   "reg_alpha": 5.6892790299999745e-08,
   "gamma": 0.1736314975486576
  }
 }
}
for k, v in PARAMS.items():
    (W / 'params' / f'{k}.json').write_text(json.dumps(v))
print(write_progress())
assert run_step(0) == 0, 'feature step failed: see the log above'

In [ ]:
# models: a failed model is skipped and the blend uses the rest
for i in range(1, len(PLAN) - 1):
    run_step(i)

In [ ]:
assert run_step(len(PLAN) - 1) == 0, 'blend failed: see the log above'

print(write_progress(final=True))
f = os.path.join(WD, 'work', 'results.jsonl')
if os.path.exists(f):
    rows = sorted((json.loads(l) for l in open(f)), key=lambda r: r['cv'])
    print('Single models, best first:')
    for r in rows:
        print(f"  {r['cv']:.5f}  {r['name']}")
sub = os.path.join(WD, 'submission.csv')
if os.path.exists(sub):
    import pandas as pd
    s = pd.read_csv(sub)
    print('\nsubmission.csv:', s.shape, '| rows sum to 1:', bool(((s.iloc[:, 1:].sum(1) - 1).abs() < 1e-6).all()))
    print(s.head(3).to_string(index=False))
total = time.time() - T0
print(f'\nTOTAL RUNTIME {mm(total)} ' + ('(OK, under the 3-hour limit)' if total < 3 * 3600 else '(OVER the 3-hour limit!)'))